# Tables 1 and 2 (clinical characteristics)

Descriptive comparisons for Tables 1 and 2. The input REDCap export (`data/clinical/bql redcap 25oct24 SOLO NOVOGENE.sav`) contains confidential clinical data and is **not** included in the repository (the folder is git-ignored). Run from `notebooks/`; outputs are written to `tables/Table_1.csv` and `tables/Table_2.csv`.

**Revision 1 (peer review):** continuous variables are now summarised with the Hodges–Lehmann location shift and its 95% CI (consistent with the Mann–Whitney U test), and all numbers use decimal points (Reviewer 2, comment 4). Software: Python 3.12, pandas, NumPy, SciPy, statsmodels, pyreadstat.


In [1]:
import pyreadstat
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import Table2x2


In [2]:
def hodges_lehmann(x, y, conf=0.95):
    """Hodges-Lehmann estimate of the shift (x - y) with its distribution-free CI."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    diffs = np.sort((x[:, None] - y[None, :]).ravel())
    n1, n2 = len(x), len(y)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    k = int(np.floor(n1 * n2 / 2 - z * np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)))
    k = max(k, 0)
    return np.median(diffs), diffs[k], diffs[n1 * n2 - k - 1]


def format_p(p):
    """[REVISION, Reviewer 2 comment 4] Consistent decimal points; 3 decimals, <0.001."""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def compare_two_groups(
    df,
    group_col,
    group_labels,
    variables,
    continuous_vars
):

    results = []

    g1, g0 = group_labels
    df1 = df[df[group_col] == 1]
    df0 = df[df[group_col] == 0]

    for var in variables:

        row = {"Variable": var}

        if var in continuous_vars:
            x1 = df1[var].dropna()
            x0 = df0[var].dropna()

            med1, q1_1, q3_1 = np.median(x1), np.percentile(x1, 25), np.percentile(x1, 75)
            med0, q1_0, q3_0 = np.median(x0), np.percentile(x0, 25), np.percentile(x0, 75)

            row[g1] = f"{med1:.2f} ({q1_1:.2f} - {q3_1:.2f})"
            row[g0] = f"{med0:.2f} ({q1_0:.2f} - {q3_0:.2f})"

            # [REVISION, Reviewer 2 comment 4] Hodges-Lehmann estimate of the location shift and its
            # 95% CI (Moses/normal approximation), consistent with the Mann-Whitney U test.
            # Replaces the bootstrap percentile CI of the difference in medians, which estimates a
            # different quantity and could include zero while the Mann-Whitney test was significant.
            hl, ci_low, ci_high = hodges_lehmann(x1, x0)

            row["HL / OR"] = f"{hl:.2f}"
            row["95% CI"] = f"({ci_low:.2f}, {ci_high:.2f})"

            _, p = stats.mannwhitneyu(x1, x0, alternative="two-sided")
            row["p-value"] = format_p(p)

        else:
            # Categórica binaria
            a = (df1[var] == 1).sum()
            b = (df1[var] == 0).sum()
            c = (df0[var] == 1).sum()
            d = (df0[var] == 0).sum()

            n1, n0 = a + b, c + d

            row[g1] = f"{100*a/n1:.1f}% ({a})" if n1 > 0 else "-"
            row[g0] = f"{100*c/n0:.1f}% ({c})" if n0 > 0 else "-"

            table = np.array([[a, b], [c, d]])

            # Fisher exact test ALWAYS
            _, p = stats.fisher_exact(table)

            if table.min() == 0:
                row["HL / OR"] = "-"
                row["95% CI"] = "-"
            else:
                t = Table2x2(table)
                row["HL / OR"] = f"{t.oddsratio:.2f}"
                ci_low, ci_high = t.oddsratio_confint()
                row["95% CI"] = f"({ci_low:.2f}, {ci_high:.2f})"

            row["p-value"] = format_p(p)

        results.append(row)

    return pd.DataFrame(results)



In [3]:
df, meta = pyreadstat.read_sav("../data/clinical/bql redcap 25oct24 SOLO NOVOGENE.sav")
df.drop("nhc",axis=1,inplace=True)

In [4]:
rename_dict = {
    "EdadIngresoOConsultaMESES": "Age (Months)",
    "sexo": "Sex (Male)",
    "lm_siono_1a": "Breastfeeding",
    "durlmtotalmeses_1a": "Breastfeeding Duration",
    "Antecedentes1gradoatopia": "Family history atopy",
    "tabaquismopasivo_siono": "Passive smoking",
    "cesarea_siono": "Cesarean section",
    "atbprevio_siono": "Previous antibiotics",
    "casocontrol": "Bronchiolitis",
    "coinfecc": "Viral co-infection",
    "rv_siono": "Rhinovirus",
    "vrs_siono": "Respiratory syncytial virus",
    "fiebre_siono": "Fever",
    "hipoxia_siono": "Hypoxia",
    "uci_siono": "ICU",
    "altoflujo_siono": "High-flow oxygen",
    "salbutamol_siono_1a": "Wheezing episodes",
    "ingrespi_siono_1a": "Respiratory hospitalization",
    "ttoasma_1a": "Asthma treatment",
}


In [5]:
vals = [val for val in rename_dict.values()] + ["Other Viruses" ]

vals = ['Age (Months)',
 'Sex (Male)',
 'Breastfeeding',
 'Breastfeeding Duration',
 'Family history atopy',
 'Passive smoking',
 'Cesarean section',
 'Previous antibiotics',
 'Bronchiolitis',
 'Viral co-infection',
 'Rhinovirus',
 'Respiratory syncytial virus',
 'Other Viruses',
 'Fever',
 'Hypoxia',
 'ICU',
 'High-flow oxygen',
 'Wheezing episodes',
 'Respiratory hospitalization',
 'Asthma treatment',
]
vals

['Age (Months)',
 'Sex (Male)',
 'Breastfeeding',
 'Breastfeeding Duration',
 'Family history atopy',
 'Passive smoking',
 'Cesarean section',
 'Previous antibiotics',
 'Bronchiolitis',
 'Viral co-infection',
 'Rhinovirus',
 'Respiratory syncytial virus',
 'Other Viruses',
 'Fever',
 'Hypoxia',
 'ICU',
 'High-flow oxygen',
 'Wheezing episodes',
 'Respiratory hospitalization',
 'Asthma treatment']

In [6]:
other_virus = ["adv_siono", "flu_siono", "mpv_siono", "piv_siono",
        "cov_siono", "ev_siono", "ttv_siono", "circovirus_siono"]

df[other_virus] = df[other_virus].replace(3,np.nan).replace(2,0)

df["Other Viruses"] = (df[other_virus]==1).any(axis=1).astype(int)

df = df.rename(columns=rename_dict)[vals]

df

,Age (Months),Sex (Male),Breastfeeding,Breastfeeding Duration,Family history atopy,Passive smoking,Cesarean section,Previous antibiotics,Bronchiolitis,Viral co-infection,Rhinovirus,Respiratory syncytial virus,Other Viruses,Fever,Hypoxia,ICU,High-flow oxygen,Wheezing episodes,Respiratory hospitalization,Asthma treatment
0,1.166667,2.0,1.0,5.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,0,1.0,1.0,2.0,2.0,1.0,2.0,2.0
1,3.333333,2.0,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,1.0,0,2.0,1.0,2.0,1.0,2.0,2.0,2.0
2,7.333333,2.0,1.0,1.5,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,0,2.0,1.0,2.0,1.0,2.0,2.0,2.0
3,2.733333,1.0,1.0,9.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1,1.0,2.0,2.0,2.0,2.0,2.0,2.0
4,5.800000,2.0,1.0,6.0,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,0,1.0,1.0,2.0,2.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,8.966667,1.0,1.0,18.0,1.0,2.0,1.0,2.0,2.0,1.0,1.0,2.0,1,NaN,NaN,NaN,NaN,1.0,2.0,2.0
86,7.833333,1.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,1.0,2.0,1.0
87,5.433333,1.0,1.0,NaN,1.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,0,NaN,NaN,NaN,NaN,2.0,2.0,2.0
88,4.133333,2.0,1.0,3.5,1.0,1.0,2.0,2.0,2.0,2.0,1.0,2.0,0,NaN,NaN,NaN,NaN,2.0,2.0,2.0


In [7]:
# [REVISION] Sex is binary and is reported as % male (Fisher test / OR), as in the manuscript tables
continous = ["Age (Months)", 'Breastfeeding Duration']
categoric = df.columns[df.columns.isin(continous)!=True].tolist()


In [8]:
df[categoric] = df[categoric].replace(3,np.nan).replace(2,0)
df

,Age (Months),Sex (Male),Breastfeeding,Breastfeeding Duration,Family history atopy,Passive smoking,Cesarean section,Previous antibiotics,Bronchiolitis,Viral co-infection,Rhinovirus,Respiratory syncytial virus,Other Viruses,Fever,Hypoxia,ICU,High-flow oxygen,Wheezing episodes,Respiratory hospitalization,Asthma treatment
0,1.166667,0.0,1.0,5.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,3.333333,0.0,1.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,7.333333,0.0,1.0,1.5,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,2.733333,1.0,1.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.800000,0.0,1.0,6.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,8.966667,1.0,1.0,18.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1,NaN,NaN,NaN,NaN,1.0,0.0,0.0
86,7.833333,1.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,1.0,0.0,1.0
87,5.433333,1.0,1.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,NaN,NaN,NaN,NaN,0.0,0.0,0.0
88,4.133333,0.0,1.0,3.5,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,NaN,NaN,NaN,NaN,0.0,0.0,0.0


In [9]:
varaibles_1 = ["Age (Months)","Sex (Male)",'Breastfeeding',
 'Breastfeeding Duration',
 'Family history atopy',
 'Passive smoking',
 'Cesarean section',
 'Previous antibiotics',
 'Viral co-infection',
 'Rhinovirus',
 'Respiratory syncytial virus',
 'Other Viruses', 'Wheezing episodes',
 'Respiratory hospitalization',
 'Asthma treatment']

In [10]:

table_1 = compare_two_groups(
    df=df,
    group_col= 'Bronchiolitis',
    group_labels=("Bronchiolitis (N=80)", "Controls (N=10)"),
    variables=varaibles_1,
    continuous_vars=continous)

table_1.to_csv("../tables/Table_1.csv", index=None)
table_1

,Variable,Bronchiolitis (N=80),Controls (N=10),HL / OR,95% CI,p-value
0,Age (Months),3.12 (1.40 - 6.01),5.15 (4.28 - 7.51),-2.50,"(-3.77, -0.23)",0.034
1,Sex (Male),57.5% (46),60.0% (6),0.90,"(0.24, 3.45)",1.000
2,Breastfeeding,82.5% (66),100.0% (9),-,-,0.344
3,Breastfeeding Duration,5.00 (2.00 - 12.00),9.00 (3.88 - 15.00),-2.00,"(-8.00, 2.00)",0.221
4,Family history atopy,67.1% (53),90.0% (9),0.23,"(0.03, 1.88)",0.272
5,Passive smoking,27.5% (22),10.0% (1),3.41,"(0.41, 28.54)",0.442
6,Cesarean section,32.9% (26),10.0% (1),4.42,"(0.53, 36.73)",0.272
7,Previous antibiotics,8.8% (7),0.0% (0),-,-,1.000
8,Viral co-infection,31.2% (24),16.7% (1),2.26,"(0.25, 20.44)",0.663
9,Rhinovirus,22.4% (17),66.7% (4),0.14,"(0.02, 0.86)",0.035


In [11]:

variables_2 = ["Age (Months)","Sex (Male)",'Breastfeeding',
 'Breastfeeding Duration',
 'Family history atopy',
 'Passive smoking',
 'Cesarean section',
 'Previous antibiotics',
 'Viral co-infection',
 'Rhinovirus',
 'Respiratory syncytial virus',
 'Other Viruses', 
 'Fever',
 'Hypoxia',
 'ICU',
 'High-flow oxygen',
 'Respiratory hospitalization',
 'Asthma treatment']


df_2= df[df.Bronchiolitis==1]

df_2

,Age (Months),Sex (Male),Breastfeeding,Breastfeeding Duration,Family history atopy,Passive smoking,Cesarean section,Previous antibiotics,Bronchiolitis,Viral co-infection,Rhinovirus,Respiratory syncytial virus,Other Viruses,Fever,Hypoxia,ICU,High-flow oxygen,Wheezing episodes,Respiratory hospitalization,Asthma treatment
0,1.166667,0.0,1.0,5.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,3.333333,0.0,1.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,7.333333,0.0,1.0,1.5,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,2.733333,1.0,1.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.800000,0.0,1.0,6.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.466667,1.0,1.0,5.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1,0.0,1.0,0.0,1.0,1.0,0.0,0.0
76,11.966667,1.0,0.0,NaN,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
77,3.466667,1.0,1.0,11.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
78,1.200000,1.0,1.0,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1.0,1.0,1.0,1.0,1.0,0.0


In [12]:
df_2["Wheezing episodes"].value_counts()

Wheezing episodes
1.0    50
0.0    30
Name: count, dtype: int64

In [13]:
table = compare_two_groups(
    df=df_2,
    group_col="Wheezing episodes",
    group_labels=("Wheezing episodes (N=50)", "No wheezing episodes (N=30)"),
    variables=variables_2,
    continuous_vars=continous)

table.to_csv("../tables/Table_2.csv", index=None)
table

,Variable,Wheezing episodes (N=50),No wheezing episodes (N=30),HL / OR,95% CI,p-value
0,Age (Months),2.43 (0.99 - 4.87),4.18 (2.33 - 6.24),-1.45,"(-2.60, -0.10)",0.037
1,Sex (Male),64.0% (32),46.7% (14),2.03,"(0.81, 5.10)",0.163
2,Breastfeeding,86.0% (43),76.7% (23),1.87,"(0.58, 5.99)",0.365
3,Breastfeeding Duration,4.00 (2.00 - 12.50),8.00 (2.00 - 12.00),-1.00,"(-5.00, 2.00)",0.478
4,Family history atopy,69.4% (34),63.3% (19),1.31,"(0.50, 3.43)",0.627
5,Passive smoking,30.0% (15),23.3% (7),1.41,"(0.50, 3.98)",0.610
6,Cesarean section,36.0% (18),27.6% (8),1.48,"(0.54, 4.01)",0.470
7,Previous antibiotics,8.0% (4),10.0% (3),0.78,"(0.16, 3.76)",1.000
8,Viral co-infection,32.7% (16),28.6% (8),1.21,"(0.44, 3.34)",0.801
9,Rhinovirus,26.5% (13),14.8% (4),2.08,"(0.60, 7.15)",0.389
